Stage 1: Generating the calibrated data from the MeerKLASS pilot pilot survey
Author: Brandon Engelbrecht

NOTE:
    For T_el we had to use JY code for it

## Important
- got the constant calculations
- has a local original copy in file for plotting purposes

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
%reload_ext autoreload

In [3]:
# Modules Required to import 

import numpy as np
import pickle
import sys
from os import path
import os
from tqdm.notebook import tqdm
import numpy.ma as ma
from scipy.interpolate import Rbf
import matplotlib.pyplot as plt


# Personal file
from satellite_RFI.src.data_reduction import *

In [4]:
"""
NOTE:
File names - select the filename for which you would like to run; 
file names have unique paths
"""

fname = '1551055211'    # 04/04/2019

folder_2018 = ['1551037708', '1551055211', '1553966342', '1554156377']

folder_2019 = ['1555775533','1555793534', '1555861810', '1556034219',
               '1556052116', '1556120503', '1556138397', '1555879611', '1561650779']

mask_level = '6'

data_save = '/idia/projects/hi_im/brandon/'+fname+'_level'+mask_level+'_mask/'

#--------------------------------
if not os.path.exists(data_save):
    print 'Path does not exist, making....'
    os.makedirs(data_save)
else:
    print 'Path exists :)'
#--------------------------------


if fname in folder_2018:
    folder = 'SCI-20180330-MS-01'
    
elif fname in folder_2019:
    folder = 'SCI-20190418-MS-01'
    
elif fname == '1562857793':
    folder = ''

dat1 = Data_reduction(file_name = fname, folder_name = folder, folder_output=data_save)

Path exists :)


/anaconda2/lib/python2.7/site-packages/katsdptelstate/telescope_state.py:338: FutureWarning: The telescope state contains pickled values. This is a security risk, but is allowed because MeerKAT data up to March 2019 uses it. You can suppress this warning by setting KATSDPTELSTATE_ALLOW_PICKLE=1 in the environment, or disable pickles by setting KATSDPTELSTATE_ALLOW_PICKLE=0.
  warnings.warn(PICKLE_WARNING, FutureWarning)


In [5]:
"""
NOTE:
Checking if the KATDAL information is already available, if not running the process to obtain

**{Does take some time to run}
"""


if path.exists(data_save+fname+'_katdal_info.p'):
    print 'Files already exists, data is pulled'
    katdal_info = pickle.load(open(data_save+fname+'_katdal_info.p', 'rb'))

    nd_s, nd_s0, freqs, nd_s0_pos, timestamps, nd_s0_coords = [katdal_info[i]
                                                               for i in katdal_info.keys()]

else:
    nd_s0, nd_s0_pos, nd_s, nd_s0_coords = dat1.get_nd_times()

    freqs, timestamps = dat1.freqs, dat1.timestamps

    # Pickling the inforamation from dat1 been pickles
    katdal_info = {"nd_s0":nd_s0, 
                   "nd_s0_pos":nd_s0_pos,
                   "nd_s":nd_s,
                   "nd_s0_coords":nd_s0_coords,
                   "frequency":freqs,
                   "time":timestamps}

    pickle.dump(katdal_info, open(data_save+fname+'_katdal_info.p', 'wb'))

Files already exists, data is pulled


In [6]:
# Frequency channels  
channel_start_idx = 600   # @ 980MHz
channel_end_idx = 3082    # @ 1500MHz
freq_band = freqs[channel_start_idx:channel_end_idx]


In [7]:
if path.exists(data_save+dat1.file_name+'_good_antennae.npy'):
    print ('Path exist')
    good_antennae = np.load(data_save+dat1.file_name+'_good_antennae.npy')
else:
    print ('Path does not exist')

Path exist


In [8]:
'''
Background Model - looks at 3 componants. The receiver, elevation and galactic.
    - Looking at different masking levels 4 or 6

'''
loc = '/idia/projects/hi_im/raw_vis/katcali_output/'    # Location of the mask data

def background_model(file_name, antenna_info, channenl_info, mask_level=None):
    '''
    file_name - fname
    antenna_info - list of antenna name
    channel - a list of the start and end channel values
    
    returns
    t_rec, t_gal, t_map, d4_mask, good_antenna list
    '''

    t_rec_h, t_rec_v = [], []
    t_gal_h, t_gal_v = [], []
    t_map_h, t_map_v = [], []
    good_antenna = []

    mask_list = [] 
    
    channel_start_idx, channel_end_idx = channenl_info
    # Loop that runs through all the t_rec info
    print ('Mask level='+mask_level)
    for i in tqdm(range(len(antenna_info)), unit='antenna'):

        # checking to see if the masks exist or not and re-writting the code
        try:
            antenna = str(antenna_info[i])[0:4]
            # Lvl 3 mask
            d3_h = pickle.load(open(loc+'level3_output/'+str(file_name)+'_'+antenna+'h_level3_data'))
            d3_v = pickle.load(open(loc+'level3_output/'+str(file_name)+'_'+antenna+'v_level3_data'))

        except IOError:
            print 'Antenna - '+antenna+' Lvl-3 data missing'
            continue
            
        try:
            if mask_level == 'l4':
                # Lvl 4 mask
                d4_mask = pickle.load(open(loc+'level4_output/mask/'+str(file_name)+'_'+antenna+'_level4_mask'))
                mask_list.append(d4_mask['Inten_mask'][nd_s0_pos, channel_start_idx:channel_end_idx])
                
            elif mask_level == 'l6':
                # Lvl 6 mask
                pix_deg=0.3
                std_sigma=2.5
                i_iter=2
                mask6_loc = '/idia/projects/hi_im/raw_vis/katcali_output/level6_output/p'+str(pix_deg)+'d/p'+str(pix_deg)+'d_sigma'+str(std_sigma)+'_iter'+str(i_iter)+'/mask/' 

                m=pickle.load(open(mask6_loc+str(fname)+'_'+str(antenna)+'_level6_p'+str(pix_deg)+'d_sigma'+str(std_sigma)+'_iter'+str(i_iter)+'_mask'))
                ch_mask=m['ch_mask']
                mask_list.append(ch_mask[channel_start_idx:channel_end_idx])
                
            else:
                print ('No mask given or incorrect mask, ony (l4) or (l6)')
            


        except IOError:
            print 'Antenna - '+antenna+' l4 or l6 mask missing'

            continue

        # T_rec information
        t_rec_h.append(d3_h['Tsm_map'][nd_s0_pos, channel_start_idx:channel_end_idx])
        t_rec_v.append(d3_v['Tsm_map'][nd_s0_pos, channel_start_idx:channel_end_idx])

        # T_gal information
        t_gal_h.append(d3_h['Tgal_map'][nd_s0_pos, channel_start_idx:channel_end_idx])
        t_gal_v.append(d3_v['Tgal_map'][nd_s0_pos, channel_start_idx:channel_end_idx])

        # T_map information
        t_map_h.append(d3_h['T_map'][nd_s0_pos, channel_start_idx:channel_end_idx])
        t_map_v.append(d3_v['T_map'][nd_s0_pos, channel_start_idx:channel_end_idx])

        # Good antenna
        good_antenna.append([int(i), antenna])

    t_rec_h = np.ma.array(t_rec_h)
    t_rec_v = np.ma.array(t_rec_v)

    t_gal_h = np.ma.array(t_gal_h)
    t_gal_v = np.ma.array(t_gal_v)

    t_map_h = np.ma.array(t_map_h)
    t_map_v = np.ma.array(t_map_v)

    mask = np.array(mask_list)
    good_antenna = np.array(good_antenna)
    
    return [t_rec_h, t_rec_v], [t_gal_h, t_gal_v], [t_map_h, t_map_v], mask, good_antenna

In [9]:
# Runs the above code
t_rec, t_gal, t_map, d_mask, good_antenna = background_model(file_name=dat1.file_name, 
                                                              antenna_info=dat1.obs_data.ants, 
                                                              channenl_info=[channel_start_idx, channel_end_idx], mask_level='l6')

Mask level=l6


Antenna - m025 Lvl-3 data missing
Antenna - m041 Lvl-3 data missing
Antenna - m042 l4 or l6 mask missing
Antenna - m051 l4 or l6 mask missing



In [10]:
# Saving good antennae
if path.exists(data_save+dat1.file_name+'_good_antennae.npy'):
    print ('Path exist')
    good_antenna = np.load(data_save+dat1.file_name+'_good_antennae.npy')
else:
    print ('Path does not exist')
    np.save(file=data_save+dat1.file_name+'_good_antennae.npy', arr=good_antenna, allow_pickle=True)

Path exist


## T-receiver

In [11]:
def t_receiver(t_rec_in, data_mask, frequency, smoothing):
    '''
    Find the t_rec for each individual antenna
    t_rec_in - input t_rec in ethier polarization
    data_mask - the mask given at the level 4 or 6 stage
    frequency - input frequency band already cgannel
    smoothing - level of smoothing done in the RBF
    '''
    # Averaging across frequency
    t_rec_across_t = np.ma.mean(t_rec_in, axis=1)   

    if data_mask.ndim == 2:
        t_rec_m = np.ma.masked_array(t_rec_in, mask=data_mask)
        t_rec_m = np.ma.array(t_rec_m)
        
    elif data_mask.ndim == 1:

        for ch_i in range(len(data_mask)):
            if data_mask[ch_i]==True:
                t_rec_in.mask[:,ch_i]=True
            t_rec_m = np.ma.array(t_rec_in)


    # Getting the frequency list for RBF interpolation
    t_rec_m_freq = np.ma.mean(t_rec_m, axis=0)

    '''Change up, switching from using the first timestamp to using the average timestamp'''
    freq_t_rec_idx = np.where(t_rec_m_freq.mask==False)[0]

    # Using RBF-linear as it seems to be the best
    '''Reason for linear can be found in JNote version 1 calculations'''
    rbf_lin = Rbf(frequency[freq_t_rec_idx], 
                    t_rec_m_freq[freq_t_rec_idx], function='linear', smooth=smoothing)

    # Interpolating over the full frequency band
    t_rec_f_interp = rbf_lin(frequency)[None, :]

    # Setting up the time and normalizing it
    t_rec_across_t_norm = t_rec_across_t[:, None] / np.max(t_rec_across_t)

    # New TOD interpolated
    t_rec_TOD = t_rec_across_t_norm * t_rec_f_interp
    
    return t_rec_TOD, freq_t_rec_idx


In [12]:
smoothing_level = 10
good_antenna_new = []
if path.exists(data_save+fname+'_S1_m000_t_rec.p'):
    print ('Path Exists \nCan pickle in the info')

else:

    for i, ant in tqdm(enumerate(good_antenna)):
        try:
            t_rec_h_interp, idx_h = t_receiver(t_rec_in=t_rec[0][i], data_mask=d_mask[i], frequency=freq_band, smoothing=smoothing_level)
            t_rec_v_interp, idx_v = t_receiver(t_rec_in=t_rec[1][i], data_mask=d_mask[i], frequency=freq_band, smoothing=smoothing_level)
            good_antenna_new.append(ant)
        except ZeroDivisionError:
                print i, ant
                print ' Bad, removing from good antenna'
                new_good_antenna = np.delete(arr=good_antenna, obj=i, axis=0)
                continue
        except ValueError:
                print i, ant
                print ' Bad, removing from good antenna'
                new_good_antenna = np.delete(arr=good_antenna, obj=i, axis=0)
                continue


        t_rec_dic = {}

        t_rec_dic['h-pol'] = t_rec[0][i]
        t_rec_dic['h-pol_interp'] = t_rec_h_interp
        t_rec_dic['v-pol'] = t_rec[1][i]
        t_rec_dic['v-pol_interp'] = t_rec_v_interp

        t_rec_dic['h-pol idx'] = idx_h
        t_rec_dic['v-pol idx'] = idx_v

        fs=open(data_save+fname+'_S1_'+ant[1]+'_t_rec.p','wb')
        pickle.dump(t_rec_dic, fs, protocol=pickle.HIGHEST_PROTOCOL)
        fs.close()

    t_rec, t_rec_h_interp, t_rec_v_interp, t_rec_dic = 0, 0, 0, 0
    good_antenna_new = np.array(good_antenna_new)
    np.save(file=data_save+dat1.file_name+'_good_antennae.npy', arr=good_antenna_new, allow_pickle=True)
    
good_antenna = np.load(data_save+dat1.file_name+'_good_antennae.npy')


Path Exists 
Can pickle in the info


## T-galactic

In [13]:
if path.exists(data_save+fname+'_S1_m000_t_gal.p'):
    print ('Path Exists')

else:
    print ('Path does not exist, conjuring')

    for i, ant in tqdm(enumerate(good_antenna)):
        t_gal_dic = {}

        t_gal_dic['h-pol'] = t_gal[0][i]
        t_gal_dic['v-pol'] = t_gal[1][i]

        fs=open(data_save+fname+'_S1_'+ant[1]+'_t_gal.p','wb')
        pickle.dump(t_gal_dic, fs, protocol=pickle.HIGHEST_PROTOCOL)
        fs.close()

Path Exists


## T-elevation

In [14]:
def temp_elevation(t_el_input):
    threshold_no = 2
    
    try:
        s_point = np.where(abs(t_el_input[0,1:] - t_el_input[0,0:-1]) > threshold_no)[0][0]
        e_point = np.where(abs(t_el_input[0,1:] - t_el_input[0,0:-1]) > threshold_no)[0][-1]

        area_of_interest = np.arange(s_point, e_point+1, 1)

        # Making a mask for the area
        masking_area = np.zeros(t_el_input.shape)
        masking_area[:, area_of_interest] = 1
        
        # New T_el
        t_le_masked = ma.masked_array(t_el_input, mask=masking_area)

        # Deleting area of interests
        t_le_masked_across_t = np.delete(t_le_masked[0], area_of_interest)
        nd_s0_2 = np.delete(nd_s0, area_of_interest)
        
        
        # Deleting area of interests
        t_le_masked_across_t = np.delete(t_le_masked[0], area_of_interest)
        nd_s0_2 = np.delete(nd_s0, area_of_interest)

        # RBF  in the temporal 
        rbf_tle_time = Rbf(nd_s0_2, t_le_masked_across_t)

        # The frequency dependance, nomralized
        tle_freq = t_le_masked[:, 0] / np.ma.max(t_le_masked[:, 0])

        # Final T_le
        t_el_final = rbf_tle_time(nd_s0)[:, None] * tle_freq[None, :]    #[K]
        print ('Data has spikes')

    except IndexError:
#         print ('Data has no spikes')

        # Final T_le
        t_el_final = t_el_input     #[K]
        
    return t_el_final.T

        

In [15]:
def t_elevation(file_name, loc, antenna, channel_info, nd_off):
    
    t_el_h, t_el_v, t_el_h_interp, t_el_v_interp  = [], [], [], []
    channel_start_idx, channel_end_idx = channel_info
    
    for i, ant in tqdm(enumerate(antenna)):
        d_h = pickle.load(open(loc+file_name+'_'+ant[1]+'h_full_t_el', 'rb'))
        d_v = pickle.load(open(loc+file_name+'_'+ant[1]+'v_full_t_el', 'rb'))
        
        t_el_h.append(d_h['Tel_map'][channel_start_idx:channel_end_idx, nd_off].T)
        t_el_v.append(d_v['Tel_map'][channel_start_idx:channel_end_idx, nd_off].T)
        
        t_el_h_interp.append(temp_elevation(t_el_input=d_h['Tel_map'][channel_start_idx:channel_end_idx, nd_off]))
        t_el_v_interp.append(temp_elevation(t_el_input=d_v['Tel_map'][channel_start_idx:channel_end_idx, nd_off]))
        
        t_el_dic = {}

        t_el_dic['h-pol'] = np.ma.array(t_el_h[i])
        t_el_dic['h-pol_interp'] = np.ma.array(t_el_h_interp[i])
        t_el_dic['v-pol'] = np.ma.array(t_el_v[i])
        t_el_dic['v-pol_interp'] = np.ma.array(t_el_v_interp[i])

        fs=open(data_save+fname+'_S1_'+ant[1]+'_t_el.p','wb')
        pickle.dump(t_el_dic, fs, protocol=pickle.HIGHEST_PROTOCOL)
        fs.close()
        
    t_el_h, t_el_v, t_el_h_interp, t_el_v_interp, t_el_dic  = 0, 0, 0, 0, 0


In [16]:
if path.exists(data_save+fname+'_S1_m000_t_el.p'):
    print ('Path Exists')

else:
    tel_loc = '/idia/projects/hi_im/brandon/T_elevation_jobs/'

    t_elevation(file_name=dat1.file_name, loc=tel_loc, antenna=good_antenna, channel_info=[channel_start_idx, channel_end_idx], nd_off=nd_s0_pos)


Path Exists


## Background model

In [17]:
ant_choice = good_antenna

In [18]:
t_noise_h = dat1.get_background_models(antenna=ant_choice[0, 1], pol='h', mask_loc=data_save)
t_noise_v = dat1.get_background_models(antenna=ant_choice[0, 1], pol='v', mask_loc=data_save)

In [19]:
# Checking if the masks are all the same
assert((t_noise_v[0].mask==t_noise_v[1].mask)).all()

#### Code deteremines the missing residual value that should be added to the background model


In [20]:
def background_model_residual():
    '''
    Determines the missing residual value 
    
    '''
    temp_residual_list = []
    background_model_list = []
    temp_tod_nu_list = []
    temp_final_residual = []

    for i in tqdm(good_antenna):
        temp_tod, bandpass, temp_tod_pol, gain_pol, bg_model, rfi_exclusion = dat1.TOD(ant_no=int(i[0]), 
                                                                                       nd_off=nd_s0_pos, 
                                                                                       mask_loc=data_save,
                                                                                       c_start=channel_start_idx, 
                                                                                       frequency=None, c_end=channel_end_idx)
        
#         print i
        
#     temp_tod.mask[1000:1500, :] = True

        fs=open(data_save+fname+'_S1_'+i[1]+'_tod.p','wb')
        pickle.dump(temp_tod, fs, protocol=pickle.HIGHEST_PROTOCOL)
        fs.close()
        
        
        # TOD temperature function of frequency
        temp_tod_nu = np.ma.mean(temp_tod, axis=0)
        temp_tod_nu_list.append(temp_tod_nu)

        # Background model
        background_model_total = (bg_model[0]+bg_model[1])/2
        background_model_list.append(background_model_total)

        temp_residual = np.ma.mean((temp_tod_nu - background_model_total)[rfi_exclusion])
        temp_residual_list.append(temp_residual)

        temp_final_residual.append(temp_tod_nu - background_model_total - temp_residual)

    temp_tod_nu, background_model_total, temp_residual = 0, 0, 0
    
    return np.array(temp_tod_nu_list), np.array(background_model_list), np.array(temp_residual_list), np.array(temp_final_residual)

In [21]:
"""
Calculating the background model in the frequency
"""

temp_tod_nu_all, background_model_all, temp_residual_all, temp_final_diff_all = background_model_residual()


In [22]:
dic = {}
dic['tod_nu'] = temp_tod_nu_all
dic['background_model_all'] = background_model_all
dic['constants'] = temp_residual_all
dic['final_diff'] = temp_final_diff_all



fs = open(data_save+dat1.file_name+'_before_weird_ant.p', 'wb')
pickle.dump(dic, fs, protocol=pickle.HIGHEST_PROTOCOL)
fs.close()

BG model per antenna

In [23]:
"""
Calcualting the background model in the 2D space
"""

def background_model_no_constant(location, antenna):
    """
    Addig the ackground models for the individual antennae together 
    location - directory of files
    antenna - name of antenna
    """
    # Elevation temperature
    tel_open = pickle.load(open(location+dat1.file_name+'_S1_'+antenna+'_t_el.p'))
    tel_sum = (tel_open['h-pol'] + tel_open['v-pol']) /2 

    # Receiver temperature
    trec_open = pickle.load(open(location+dat1.file_name+'_S1_'+antenna+'_t_rec.p'))
    trec_sum = (trec_open['h-pol_interp'] + trec_open['v-pol_interp']) / 2

    # Galactic temperature
    tgal_open = pickle.load(open(location+dat1.file_name+'_S1_'+antenna+'_t_gal.p'))
    tgal_sum = (tgal_open['h-pol'] + tgal_open['v-pol']) / 2

    # CMB Temperature
    tcmb = 2.73

    # Summing Temperature
    return tel_sum + trec_sum + tgal_sum + tcmb

In [24]:
"""
Adding the Background Models to the data
"""
for i, ant in enumerate(tqdm(good_antenna)):
    fs = open(data_save+dat1.file_name+'_S1_'+ant[1]+'_bg_model_no_con.p', 'wb')
    bg = background_model_no_constant(location = data_save, antenna = ant[1])
    pickle.dump(bg, fs, protocol=pickle.HIGHEST_PROTOCOL)
    fs.close()

Removing weird antennae

In [25]:
# Looking for antennae that have large constant values
weird_antenna = np.where(abs((temp_residual_all)) > np.median(abs(temp_residual_all)) * 10)[0]

In [26]:
def delete_weird_antenna(w_antenna, g_antenna=good_antenna, tod_all=temp_tod_nu_all, 
                         bg_model=background_model_all, residuals=temp_residual_all, final_dff=temp_final_diff_all):
    '''
    Removes the the weird antenna from the list of values, requires the antenna in list or array
    '''
    good_ant = np.delete(arr=g_antenna, obj=w_antenna, axis=0)
    w = np.delete(arr=tod_all, obj=w_antenna, axis=0)
    x = np.delete(arr=bg_model, obj=w_antenna, axis=0)
    y = np.delete(arr=residuals, obj=w_antenna, axis=None)
    z = np.delete(arr=final_dff, obj=w_antenna, axis=0)
        
    return w, x, y, z, good_ant

In [27]:
temp_tod_nu, background_model, temp_residual, temp_diff, good_antenna_n = delete_weird_antenna(w_antenna=weird_antenna)

In [28]:
dic = {}
dic['tod_nu'] = temp_tod_nu
dic['background_model_all'] = background_model
dic['constants'] = temp_residual
dic['final_diff'] = temp_diff
dic['antenna'] = good_antenna_n[:,1]


# Saving the constansts that do not need to be deleted
fs = open(data_save+dat1.file_name+'_after_weird_ant.p', 'wb')
pickle.dump(dic, fs, protocol=pickle.HIGHEST_PROTOCOL)
fs.close()

Averaging the antennae together

In [29]:
tod_avg = 0
bg_model_avg = 0

for i, ant in tqdm(enumerate(good_antenna_n)):
    tod_avg += pickle.load(open(data_save+dat1.file_name+'_S1_'+ant[1]+'_tod.p', 'rb'))
        
    bg_model_avg += pickle.load(open(data_save+dat1.file_name+'_S1_'+ant[1]+'_bg_model_no_con.p', 'rb')) + temp_residual[i]
    
dic = {}
dic['TOD Avg'] = tod_avg/i
dic['BG Model'] = bg_model_avg/i

fs = open(data_save+dat1.file_name+'_average_TOD_BG_model.p', 'wb')
pickle.dump(dic, fs, protocol=pickle.HIGHEST_PROTOCOL)
fs.close()

## ----------------------------------------------------------END-----------------------------------------------------------